# Load Rate Prediction – Machine Learning Assessment

**Objective:** Predict `posted_rate` for freight loads.

- Train data: 48,000 loads (Jan – Oct 2025)
- Validation data: 12,000 loads (Nov – Dec 2025)
- Model: XGBoost Regressor
- Validation strategy: Time-based split

## 1. Import Libraries

In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error
import xgboost as xgb
import warnings
warnings.filterwarnings("ignore")

## 2. Load Data

In [3]:
train = pd.read_csv("/Users/Zhuanz/Desktop/Machine Learning Engineer | Assessment/train-test.csv")
val = pd.read_csv("/Users/Zhuanz/Desktop/Machine Learning Engineer | Assessment/validation.csv")
template = pd.read_csv("/Users/Zhuanz/Desktop/Machine Learning Engineer | Assessment/validation-predictions-template.csv")

train["date"] = pd.to_datetime(train["date"])
val["date"] = pd.to_datetime(val["date"])

print("Train shape:", train.shape)
print("Validation shape:", val.shape)
print("\nColumns:", train.columns.tolist())

Train shape: (48000, 14)
Validation shape: (12000, 13)

Columns: ['load_id', 'pickup', 'delivery', 'pickup_lat', 'pickup_lon', 'delivery_lat', 'delivery_lon', 'distance', 'equipment', 'weight', 'date', 'market_index', 'quote_signal', 'posted_rate']


## 3. Exploratory Data Analysis

In [4]:
# Target distribution
print("Target (posted_rate) statistics:")
print(train["posted_rate"].describe())
print("\nZero or negative rates:", (train["posted_rate"] <= 0).sum())

Target (posted_rate) statistics:
count    48000.000000
mean      2373.980682
std       1486.493245
min         57.220000
25%       1251.555000
50%       2030.760000
75%       3330.750000
max      25533.000000
Name: posted_rate, dtype: float64

Zero or negative rates: 0


In [5]:
# Date ranges
print("Date range (train):", train["date"].min(), "→", train["date"].max())
print("Date range (val):  ", val["date"].min(), "→", val["date"].max())

print("\nLoads per month:")
print(train.groupby(train["date"].dt.to_period("M")).size())

Date range (train): 2025-01-01 00:00:00 → 2025-10-31 00:00:00
Date range (val):   2025-11-01 00:00:00 → 2025-12-31 00:00:00

Loads per month:
date
2025-01    4918
2025-02    4337
2025-03    5036
2025-04    4819
2025-05    4913
2025-06    4783
2025-07    4912
2025-08    4759
2025-09    4670
2025-10    4853
Freq: M, dtype: int64


In [6]:
# Categorical cardinality
print("Unique pickups:", train["pickup"].nunique())
print("Unique deliveries:", train["delivery"].nunique())
print("Unique equipment:", train["equipment"].nunique())
print("\nEquipment distribution:")
print(train["equipment"].value_counts())

Unique pickups: 64
Unique deliveries: 64
Unique equipment: 3

Equipment distribution:
equipment
Dry Van    27202
Reefer     12045
Flatbed     8753
Name: count, dtype: int64


In [7]:
# Missing values
print("Missing values:")
print(train.isnull().sum().sort_values(ascending=False).head(10))

Missing values:
market_index    374
weight          300
load_id           0
pickup            0
delivery          0
pickup_lat        0
pickup_lon        0
delivery_lat      0
delivery_lon      0
distance          0
dtype: int64


In [8]:
# Correlation with target
num_cols = ["distance", "weight", "market_index", "quote_signal",
            "pickup_lat", "pickup_lon", "delivery_lat", "delivery_lon"]
print("Correlation with posted_rate:")
print(train[num_cols + ["posted_rate"]].corr()["posted_rate"].sort_values(ascending=False))

Correlation with posted_rate:
posted_rate     1.000000
distance        0.908519
weight          0.034840
market_index    0.034165
quote_signal   -0.039858
pickup_lat     -0.090873
delivery_lat   -0.091970
pickup_lon     -0.255058
delivery_lon   -0.257086
Name: posted_rate, dtype: float64


## 4. Feature Engineering

In [9]:
def engineer_features(df, is_train=True, te_maps=None):
    """Create model features. Fit target encodings only on training data."""
    df = df.copy()

    # Impute missing values (only weight and market_index had missing values)
    df["weight"] = df["weight"].fillna(df["weight"].median())
    df["market_index"] = df["market_index"].fillna(df["market_index"].median())

    # Distance features
    df["log_distance"] = np.log1p(df["distance"])

    # Date features
    df["month"] = df["date"].dt.month
    df["dayofweek"] = df["date"].dt.dayofweek
    df["is_weekend"] = (df["dayofweek"] >= 5).astype(int)
    df["weekofyear"] = df["date"].dt.isocalendar().week.astype(int)

    # Equipment one-hot encoding
    df = pd.get_dummies(df, columns=["equipment"], prefix="eq")

    # Target encoding for pickup & delivery locations
    if is_train:
        te_maps = {}
        global_mean = df["posted_rate"].mean()
        for col in ["pickup", "delivery"]:
            te_maps[col] = df.groupby(col)["posted_rate"].mean().to_dict()
            df[f"{col}_te"] = df[col].map(te_maps[col]).fillna(global_mean)
        return df, te_maps
    else:
        global_mean = np.mean(list(te_maps["pickup"].values()))
        for col in ["pickup", "delivery"]:
            df[f"{col}_te"] = df[col].map(te_maps[col]).fillna(global_mean)
        return df

In [10]:
train_fe, te_maps = engineer_features(train, is_train=True)
val_fe = engineer_features(val, is_train=False, te_maps=te_maps)

# Interaction features
for df in [train_fe, val_fe]:
    df["dist_x_market"] = df["distance"] * df["market_index"]
    df["dist_x_quote"] = df["distance"] * df["quote_signal"]

print("Feature engineering complete.")

Feature engineering complete.


## 5. Time-based Validation Split

In [11]:
FEATURES = [
    "distance", "log_distance", "weight", "market_index", "quote_signal",
    "month", "dayofweek", "is_weekend", "weekofyear",
    "pickup_te", "delivery_te",
    "eq_Dry Van", "eq_Reefer", "eq_Flatbed",
    "dist_x_market", "dist_x_quote",
    "pickup_lat", "pickup_lon", "delivery_lat", "delivery_lon"
]

# Ensure all columns exist
for col in FEATURES:
    if col not in train_fe.columns:
        train_fe[col] = 0
    if col not in val_fe.columns:
        val_fe[col] = 0

# Time-based split: train on Jan–Aug, validate on Sep–Oct
cutoff = pd.Timestamp("2025-08-31")
train_set = train_fe[train_fe["date"] < cutoff].copy()
valid_set = train_fe[train_fe["date"] >= cutoff].copy()

X_train = train_set[FEATURES]
y_train = train_set["posted_rate"]
X_valid = valid_set[FEATURES]
y_valid = valid_set["posted_rate"]

print(f"Train period : {train_set['date'].min().date()} → {train_set['date'].max().date()} ({len(train_set):,} rows)")
print(f"Valid period : {valid_set['date'].min().date()} → {valid_set['date'].max().date()} ({len(valid_set):,} rows)")

Train period : 2025-01-01 → 2025-08-30 (38,330 rows)
Valid period : 2025-08-31 → 2025-10-31 (9,670 rows)


## 6. Train XGBoost Model

In [12]:
model = xgb.XGBRegressor(
    n_estimators=3000,
    learning_rate=0.02,
    max_depth=8,
    min_child_weight=5,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1,
    early_stopping_rounds=150
)

model.fit(
    X_train, y_train,
    eval_set=[(X_valid, y_valid)],
    verbose=100
)

[0]	validation_0-rmse:1498.82358
[100]	validation_0-rmse:664.38457
[200]	validation_0-rmse:639.92585
[300]	validation_0-rmse:641.98200
[339]	validation_0-rmse:643.20057


,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'reg:squarederror'
,"base_score base_score: typing.Optional[float]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.85
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",150
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[str], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = loa

In [13]:
# Evaluate
preds = model.predict(X_valid)
mae = mean_absolute_error(y_valid, preds)
rmse = np.sqrt(mean_squared_error(y_valid, preds))

print(f"Validation MAE : {mae:.2f}")
print(f"Validation RMSE: {rmse:.2f}")

# Feature importance
imp = pd.Series(model.feature_importances_, index=FEATURES).sort_values(ascending=False)
print("\nFeature Importance:")
print(imp)

Validation MAE : 143.50
Validation RMSE: 639.71

Feature Importance:
log_distance     0.545887
distance         0.320761
dist_x_quote     0.025344
dist_x_market    0.013018
eq_Dry Van       0.011214
is_weekend       0.010489
dayofweek        0.006654
eq_Reefer        0.006477
month            0.005844
quote_signal     0.005788
delivery_te      0.005634
weight           0.005514
market_index     0.005239
pickup_te        0.005121
delivery_lon     0.005103
weekofyear       0.004787
pickup_lon       0.004679
delivery_lat     0.004588
pickup_lat       0.004326
eq_Flatbed       0.003532
dtype: float32


## 7. Retrain on Full Data & Generate Final Predictions

In [14]:
# Retrain on all training data (Jan–Oct)
X_full = train_fe[FEATURES]
y_full = train_fe["posted_rate"]

final_model = xgb.XGBRegressor(
    n_estimators=model.best_iteration + 50,
    learning_rate=0.02,
    max_depth=8,
    min_child_weight=5,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1
)

final_model.fit(X_full, y_full, verbose=False)
print("Final model trained on full data.")

Final model trained on full data.


In [15]:
# Predict on real validation set (Nov–Dec)
final_preds = final_model.predict(val_fe[FEATURES])

submission = template.copy()
submission["predicted_rate"] = final_preds
submission.to_csv("validation_predictions.csv", index=False)

print("Saved: validation_predictions.csv")
print(submission["predicted_rate"].describe())

Saved: validation_predictions.csv
count    12000.000000
mean      2436.949951
std       1433.824219
min        247.010757
25%       1315.481110
50%       2078.476074
75%       3484.651367
max       7795.470215
Name: predicted_rate, dtype: float64


## 8. Create December Chart Inputs (Fixed Lane)

In [16]:
december_dates = pd.date_range("2025-12-01", "2025-12-31", freq="D")

december_raw = pd.DataFrame({
    "pickup": "Lexington",
    "delivery": "Fort Wayne",
    "pickup_lat": 38.0406,
    "pickup_lon": -84.5037,
    "delivery_lat": 41.0793,
    "delivery_lon": -85.1394,
    "distance": 360.0,
    "equipment": "Dry Van",
    "weight": 32000.0,
    "date": december_dates,
    "market_index": train["market_index"].median(),
    "quote_signal": train["quote_signal"].median()
})

december_fe = engineer_features(december_raw, is_train=False, te_maps=te_maps)
december_fe["dist_x_market"] = december_fe["distance"] * december_fe["market_index"]
december_fe["dist_x_quote"] = december_fe["distance"] * december_fe["quote_signal"]

for col in FEATURES:
    if col not in december_fe.columns:
        december_fe[col] = 0

december_preds = final_model.predict(december_fe[FEATURES])

december_output = pd.DataFrame({
    "pickup": "Lexington",
    "delivery": "Fort Wayne",
    "distance": 360.0,
    "equipment": "Dry Van",
    "weight": 32000.0,
    "date": december_dates,
    "predicted_rate": december_preds
})

december_output.to_csv("december_chart_inputs.csv", index=False)
print("Saved: december_chart_inputs.csv")
print(f"December predicted rate range: {december_preds.min():.2f} → {december_preds.max():.2f}")

Saved: december_chart_inputs.csv
December predicted rate range: 827.03 → 877.77
